In [43]:
import numpy as np 
import pandas as pd
import os

In [55]:
path = os.path.join("..", "data_processed", "nazare_ocean_raw.parquet")

df_hourly = pd.read_parquet(path)


The available points should be averaged to obtain exactly one row corresponding to one hour, which will also result in one average point per grid, resulting from the arithmetic mean of both longitude and latitude. The debate over which method to choose: selecting a single real point or averaging the real points comes down to a balance between bias and variance; a 'golden mean' must be found here. In this case, the grid is quite wide, so the decision was made to average the available points.

In [56]:
df_hourly = df.groupby('valid_time').mean(numeric_only = True).reset_index()

df_hourly.head()

,valid_time,latitude,longitude,tp,number,u10,v10,d2m,t2m,msl,sst,ishf,mwd,mwp,swh,pp1d
0,2000-01-01 00:00:00,39.5,-9.55,0.0,0.0,-2.685440,-4.544415,280.510010,284.789459,102746.656250,288.472961,-17.092773,327.884796,8.214509,1.382858,9.931007
1,2000-01-01 06:00:00,39.5,-9.55,0.0,0.0,-2.519270,-3.441375,280.830597,284.573120,102722.390625,288.472961,-14.455933,323.430359,8.842020,1.305284,13.316505
2,2000-01-01 12:00:00,39.5,-9.55,0.0,0.0,-1.899942,-2.719204,279.305237,285.521851,102908.000000,288.472961,-25.772461,316.730377,10.025695,1.339482,13.428322
3,2000-01-01 18:00:00,39.5,-9.55,0.0,0.0,-1.607701,-4.807054,280.081573,284.916077,102775.320312,288.472961,-14.737640,311.361450,11.419249,1.656322,14.627785
4,2000-01-02 00:00:00,39.5,-9.55,0.0,0.0,-2.216261,-4.862338,280.305023,285.010315,102842.703125,288.487457,-17.002182,309.866638,12.182859,2.160654,14.639992


This also allows you to remove the latitude and longitude columns, while clearly indicating that the data is taken from the virtual point with coordinates [39.5, -9.55].

In [46]:
df_hourly.drop(columns = ['latitude', 'longitude'], inplace = True)

The next step is to address the number and tp (total rainfall) columns.

The number column is a placeholder added to the ERA5 data. It does not contribute any information to the model, so it should be removed.

Removing the column number:

In [47]:
df_hourly.drop(columns = ['number'], inplace = True)

### Feature Selection: Precipitation Data
Note: Initial analysis of the precipitation data ('tp' / total precipitation) showed it was either redundant or of insufficient quality (high percentage of missing/zero values) for the offshore wave height prediction. 
**Decision:** This feature has been excluded from the final analysis to avoid introducing noise into the model.

In [48]:
df_hourly.drop(columns = ['tp'], inplace = True)

In [49]:
df_hourly.head()

,valid_time,u10,v10,d2m,t2m,msl,sst,ishf,mwd,mwp,swh,pp1d
0,2000-01-01 00:00:00,-2.685440,-4.544415,280.510010,284.789459,102746.656250,288.472961,-17.092773,327.884796,8.214509,1.382858,9.931007
1,2000-01-01 06:00:00,-2.519270,-3.441375,280.830597,284.573120,102722.390625,288.472961,-14.455933,323.430359,8.842020,1.305284,13.316505
2,2000-01-01 12:00:00,-1.899942,-2.719204,279.305237,285.521851,102908.000000,288.472961,-25.772461,316.730377,10.025695,1.339482,13.428322
3,2000-01-01 18:00:00,-1.607701,-4.807054,280.081573,284.916077,102775.320312,288.472961,-14.737640,311.361450,11.419249,1.656322,14.627785
4,2000-01-02 00:00:00,-2.216261,-4.862338,280.305023,285.010315,102842.703125,288.487457,-17.002182,309.866638,12.182859,2.160654,14.639992


The final preprocessing step is converting ERA5 units, which are standardized to the SI system, to more intuitive units for analysis.

Columns requiring changes:

u10 and v10 = wind components - current unit [m/s], target unit [km/h].

sst, t2m, d2m = sea surface temperature, air temperature, dew point temperature - current unit [K], target unit [C]

msl = mean sea level pressure - current unit [Pa], target unit [hPa]

In [50]:
#Przeliczanie temperatur:
df_hourly['air_temp_c'] = df_hourly['t2m'] - 273.15
df_hourly['sea_surface_temp_c'] = df_hourly['sst'] - 273.15
df_hourly['dew_point_temp_c'] = df_hourly['d2m'] - 273.15

df_hourly['wind_speed_kmh'] = np.sqrt(df_hourly['u10']**2 + df_hourly['v10']**2) * 3.6

df_hourly['pressure_hpa'] = df_hourly['msl'] / 100



The next preprocessing step is to add a column describing the wind direction and remove unnecessary columns.

In [51]:
df_hourly['wind_direction'] = (np.degrees(np.arctan2(df_hourly['u10'], df_hourly['v10'])) + 180) % 360

def show_direction(angle):
    dirs = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    ix = int((angle + 22.5) / 45)
    return dirs[ix % 8]

df_hourly['wind_direction'] = df_hourly['wind_direction'].apply(show_direction)

cols_to_drop = ['u10', 'v10', 't2m', 'sst', 'd2m', 'msl']
df_hourly = df_hourly.drop(columns = [c for c in cols_to_drop if c in df_hourly.columns])

The final step is to extract the following from the 'valid_time' column:
- Year
- Month
- Day
- Hour

And rename the columns for intuitive analysis.

In [52]:
df_hourly['valid_time'] = pd.to_datetime(df_hourly['valid_time'])

df_hourly['year'] = df_hourly['valid_time'].dt.year
df_hourly['month'] = df_hourly['valid_time'].dt.month
df_hourly['day'] = df_hourly['valid_time'].dt.day
df_hourly['hour'] = df_hourly['valid_time'].dt.hour

In [53]:
df_hourly = df_hourly.rename(columns = {'swh': 'wave_height_m',
    'pp1d': 'peak_wave_period_s',
    'mwp': 'mean_wave_period_s',
    'mwd': 'wave_direction_deg',
    'ishf': 'surface_heat_flux'})

df_hourly.head()

,valid_time,surface_heat_flux,wave_direction_deg,mean_wave_period_s,wave_height_m,peak_wave_period_s,air_temp_c,sea_surface_temp_c,dew_point_temp_c,wind_speed_kmh,pressure_hpa,wind_direction,year,month,day,hour
0,2000-01-01 00:00:00,-17.092773,327.884796,8.214509,1.382858,9.931007,11.639465,15.322968,7.360016,19.002850,1027.466553,NE,2000,1,1,0
1,2000-01-01 06:00:00,-14.455933,323.430359,8.842020,1.305284,13.316505,11.423126,15.322968,7.680603,15.353812,1027.223877,NE,2000,1,1,6
2,2000-01-01 12:00:00,-25.772461,316.730377,10.025695,1.339482,13.428322,12.371857,15.322968,6.155243,11.941938,1029.079956,NE,2000,1,1,12
3,2000-01-01 18:00:00,-14.737640,311.361450,11.419249,1.656322,14.627785,11.766083,15.322968,6.931580,18.247585,1027.753174,N,2000,1,1,18
4,2000-01-02 00:00:00,-17.002182,309.866638,12.182859,2.160654,14.639992,11.860321,15.337463,7.155029,19.236988,1028.427002,NE,2000,1,2,0


All that remains is to save the dataset to a file

In [54]:
import pyarrow
import os

output_dir = os.path.join("..", "data_processed")
file_name = "nazare_cleaned_final.parquet"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, file_name)

df_hourly.to_parquet(file_path, index=False)